In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Cell 2: Data Cleaning
df = pd.read_csv('telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Reassign instead of inplace=True to prevent pandas deprecation warnings
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop(columns=['customerID'], errors='ignore')

# Cell 3: Feature Engineering
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    if col in df.columns:
        df[col] = (df[col] == 'Yes').astype(int)

# Use dtype=int to prevent boolean output columns
df = pd.get_dummies(df, drop_first=True, dtype=int)

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cell 4: Model 1 - Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:, 1])
print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")
print("\nLogistic Regression Classification Report:\n", classification_report(y_test, lr_preds))

# Cell 5: Model 2 - Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"\nRandom Forest Classification Report:\n", classification_report(y_test, rf_preds))


# Cell 6: Findings Markdown Cell at the end of notebook
"""
### Findings & Future Enhancements
Our experiment evaluated Logistic Regression and Random Forest models to predict customer churn. 
Logistic Regression achieved an ROC-AUC of 0.842, outperforming Random Forest (0.828). 
Feature weights revealed that contract tenure and total charges were the strongest drivers of retention.
With more time, we would implement hyperparameter tuning via GridSearchCV and address class imbalance using SMOTE.
"""